# PR merge throughput


## Preliminaries


In [ ]:
# Run me to download latest data from GitHub

from qb_notebook.artifacts import (
    download_and_extract_latest_successful_workflow_artifacts,
)

info = download_and_extract_latest_successful_workflow_artifacts(
    repo="leanprover-community/queueboard-core",
    workflow="upload_backup.yaml",
    out_dir="./data",
    artifact_name="analytics-datasets",
    branch="master",
    search_limit=100,  # change this if you expect there to be > 100 failed runs before the first successful one
)

info

In [ ]:
import polars as pl

from qb_notebook.data_io import load_pr_interval_data

tables = load_pr_interval_data("data")

df_prs = tables["prs"]

## PRs merged per day (Bors, EDA)

Exploratory analysis of daily merge throughput, following the approach in `generate_plot_site.py`: filtering `df_prs` directly to titles matching `[Merged by Bors]` and computing daily counts with rolling averages.

In [ ]:
import matplotlib.pyplot as plt
from datetime import timedelta, datetime, timezone

# Build merged-per-day from df_prs directly (matching generate_plot_site.py)
df_merged_bors = df_prs.filter(
    pl.col("closed_at").is_not_null()
    & pl.col("title")
    .fill_null("")
    .str.contains(r"(?i)^\[Merged by Bors\] -", literal=False)
)

daily_bors = (
    df_merged_bors.with_columns(pl.col("closed_at").dt.truncate("1d").alias("date"))
    .group_by("date")
    .agg(pl.len().alias("prs_merged"))
    .sort(
        "date"
    )  # must sort before rolling_mean; Polars rolling_mean is row-order-based, not time-based
    .with_columns(pl.col("prs_merged").rolling_mean(14).alias("prs_merged_14d_avg"))
)

daily_bors

In [ ]:
# Plot: raw daily count + 14-day average, since 2023-01-01
pdf = daily_bors.to_pandas()
pdf = pdf[pdf["date"] >= datetime(2023, 1, 1, tzinfo=timezone.utc)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(
    pdf["date"],
    pdf["prs_merged"],
    alpha=0.3,
    label="daily count",
    color="steelblue",
    width=1,
)
ax.plot(
    pdf["date"],
    pdf["prs_merged_14d_avg"],
    label="14-day avg",
    color="steelblue",
    linewidth=2,
)
ax.set_xlabel("date (UTC)")
ax.set_ylabel("PRs merged")
ax.set_title("PRs merged per day (title matches '[Merged by Bors]', since 2023-01-01)")
ax.legend()
fig.autofmt_xdate(rotation=45)
fig.tight_layout()
plt.show()

In [ ]:
# Plot: last 365 days
pdf = daily_bors.to_pandas()
if not pdf.empty:
    cutoff = pdf["date"].max() - timedelta(days=365)
    pdf = pdf[pdf["date"] > cutoff]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(
    pdf["date"],
    pdf["prs_merged"],
    alpha=0.3,
    label="daily count",
    color="steelblue",
    width=1,
)
ax.plot(
    pdf["date"],
    pdf["prs_merged_14d_avg"],
    label="14-day avg",
    color="steelblue",
    linewidth=2,
)
ax.set_xlabel("date (UTC)")
ax.set_ylabel("PRs merged")
ax.set_title("PRs merged per day (title matches '[Merged by Bors]', last 365 days)")
ax.legend()
fig.autofmt_xdate(rotation=45)
fig.tight_layout()
plt.show()

In [ ]:
# feat vs non-feat daily merge counts (14-day avg)
feat_expr = (
    pl.col("title")
    .fill_null("")
    .str.contains(r"(^\[Merged by Bors\] -\s+[fF]eat)", literal=False)
)


def daily_merged_counts(df, col_name):
    return (
        df.with_columns(pl.col("closed_at").dt.truncate("1d").alias("date"))
        .group_by("date")
        .agg(pl.len().alias(col_name))
        .sort("date")
    )


feat_o_expr = (
    pl.col("title")
    .fill_null("")
    .str.contains(r"([fF]eat)|(^\[Merged by Bors\] -\s+[fF]eat)", literal=False)
)


def daily_opened_counts(df, col_name):
    return (
        df.with_columns(pl.col("gh_created_at").dt.truncate("1d").alias("date"))
        .group_by("date")
        .agg(pl.len().alias(col_name))
        .sort("date")
    )


daily_feat = daily_merged_counts(df_merged_bors.filter(feat_expr), "feat")
daily_nonfeat = daily_merged_counts(df_merged_bors.filter(~feat_expr), "non_feat")

daily_feat_nonfeat = (
    daily_feat.join(daily_nonfeat, on="date", how="full", coalesce=True)
    .with_columns(pl.col("feat").fill_null(0), pl.col("non_feat").fill_null(0))
    .sort(
        "date"
    )  # must sort before rolling_mean; Polars rolling_mean is row-order-based, not time-based
    .with_columns(
        pl.col("feat").rolling_mean(14).alias("feat_14d_avg"),
        pl.col("non_feat").rolling_mean(14).alias("non_feat_14d_avg"),
    )
)

daily_o_feat = daily_opened_counts(df_prs.filter(feat_o_expr), "feat_o")
daily_o_nonfeat = daily_opened_counts(df_prs.filter(~feat_o_expr), "non_feat_o")

daily_o_feat_nonfeat = (
    daily_o_feat.join(daily_o_nonfeat, on="date", how="full", coalesce=True)
    .with_columns(pl.col("feat_o").fill_null(0), pl.col("non_feat_o").fill_null(0))
    .sort(
        "date"
    )  # must sort before rolling_mean; Polars rolling_mean is row-order-based, not time-based
    .with_columns(
        pl.col("feat_o").rolling_mean(14).alias("feat_14d_avg"),
        pl.col("non_feat_o").rolling_mean(14).alias("non_feat_14d_avg"),
    )
)

daily_o_m_feat = (
    daily_feat.join(daily_o_feat, on="date", how="full", coalesce=True)
    .with_columns(pl.col("feat").fill_null(0), pl.col("feat_o").fill_null(0))
    .sort(
        "date"
    )  # must sort before rolling_mean; Polars rolling_mean is row-order-based, not time-based
    .with_columns(
        pl.col("feat").rolling_mean(14).alias("feat_14d_avg"),
        pl.col("feat_o").rolling_mean(14).alias("feat_o_14d_avg"),
    )
)

daily_o_m_nonfeat = (
    daily_nonfeat.join(daily_o_nonfeat, on="date", how="full", coalesce=True)
    .with_columns(pl.col("non_feat").fill_null(0), pl.col("non_feat_o").fill_null(0))
    .sort(
        "date"
    )  # must sort before rolling_mean; Polars rolling_mean is row-order-based, not time-based
    .with_columns(
        pl.col("non_feat").rolling_mean(14).alias("non_feat_14d_avg"),
        pl.col("non_feat_o").rolling_mean(14).alias("non_feat_o_14d_avg"),
    )
)

daily_feat_nonfeat

In [ ]:
# Plot: feat vs non-feat 14-day avg, since 2023-01-01
pdf = daily_feat_nonfeat.to_pandas()
pdf = pdf[pdf["date"] >= datetime(2023, 1, 1, tzinfo=timezone.utc)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    pdf["date"],
    pdf["feat_14d_avg"],
    label="feat (14d avg)",
    color="#2ca02c",
    linewidth=2,
)
ax.plot(
    pdf["date"],
    pdf["non_feat_14d_avg"],
    label="non-feat (14d avg)",
    color="#ff7f0e",
    linewidth=2,
)
ax.set_xlabel("date (UTC)")
ax.set_ylabel("PRs merged per day")
ax.set_title("PRs merged per day: feat vs non-feat (14d avg, since 2023-01-01)")
ax.legend()
fig.autofmt_xdate(rotation=45)
fig.tight_layout()
plt.show()

In [ ]:
# Plot: feat vs non-feat 14-day avg, since 2023-01-01
pdf = daily_o_feat_nonfeat.to_pandas()
pdf = pdf[pdf["date"] >= datetime(2023, 1, 1, tzinfo=timezone.utc)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    pdf["date"],
    pdf["feat_14d_avg"],
    label="feat (14d avg)",
    color="#2ca02c",
    linewidth=2,
)
ax.plot(
    pdf["date"],
    pdf["non_feat_14d_avg"],
    label="non-feat (14d avg)",
    color="#ff7f0e",
    linewidth=2,
)
ax.set_xlabel("date (UTC)")
ax.set_ylabel("PRs opened per day")
ax.set_title("PRs opened per day: feat vs non-feat (14d avg, since 2023-01-01)")
ax.legend()
fig.autofmt_xdate(rotation=45)
fig.tight_layout()
plt.show()

In [ ]:
# Plot: opened feat vs merged feat 14-day avg, since 2023-01-01
pdf = daily_o_m_feat.to_pandas()
pdf = pdf[pdf["date"] >= datetime(2023, 1, 1, tzinfo=timezone.utc)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    pdf["date"],
    pdf["feat_14d_avg"],
    label="merged feat (14d avg)",
    color="#2ca02c",
    linewidth=2,
)
ax.plot(
    pdf["date"],
    pdf["feat_o_14d_avg"],
    label="opened feat (14d avg)",
    color="#0f463f",
    linewidth=2,
)
ax.set_xlabel("date (UTC)")
ax.set_ylabel("feat PRs per day")
ax.set_title("feat PRs per day: merged vs opened (14d avg, since 2023-01-01)")
ax.legend()
fig.autofmt_xdate(rotation=45)
fig.tight_layout()
plt.show()

In [ ]:
# Plot: opened non-feat vs merged non-feat 14-day avg, since 2023-01-01
pdf = daily_o_m_nonfeat.to_pandas()
pdf = pdf[pdf["date"] >= datetime(2023, 1, 1, tzinfo=timezone.utc)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    pdf["date"],
    pdf["non_feat_14d_avg"],
    label="merged non-feat (14d avg)",
    color="#ff7f0e",
    linewidth=2,
)
ax.plot(
    pdf["date"],
    pdf["non_feat_o_14d_avg"],
    label="opened non-feat (14d avg)",
    color="#744C02",
    linewidth=2,
)
ax.set_xlabel("date (UTC)")
ax.set_ylabel("non-feat PRs per day")
ax.set_title("non-feat PRs per day: merged vs opened (14d avg, since 2023-01-01)")
ax.legend()
fig.autofmt_xdate(rotation=45)
fig.tight_layout()
plt.show()

In [ ]:
# Year-over-year comparison: total merges per calendar year
yearly = (
    df_merged_bors.with_columns(pl.col("closed_at").dt.year().alias("year"))
    .group_by("year")
    .agg(pl.len().alias("prs_merged"))
    .sort("year")
)

pdf_yr = yearly.to_pandas()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(pdf_yr["year"].astype(str), pdf_yr["prs_merged"], color="steelblue")
ax.set_xlabel("year")
ax.set_ylabel("PRs merged")
ax.set_title("Total PRs merged per year (title matches '[Merged by Bors]')")
for i, (yr, cnt) in enumerate(zip(pdf_yr["year"], pdf_yr["prs_merged"])):
    ax.text(i, cnt + 5, str(cnt), ha="center", va="bottom", fontsize=9)
fig.tight_layout()
plt.show()

## PRs filtered by MI contributors

In [ ]:
from qb_notebook.data_io import load_contributor_config
from qb_notebook.filters import expr_commenters_include_any

config = load_contributor_config("contributors_MI.json")
logins = [e.login for e in config]
df_their_prs = df_merged_bors.filter(expr_commenters_include_any(logins))

mi_expr = expr_commenters_include_any(logins)

daily_mi = daily_merged_counts(df_merged_bors.filter(mi_expr), "mi")
daily_nonmi = daily_merged_counts(df_merged_bors.filter(~mi_expr), "non_mi")

daily_mi_nonmi = (
    daily_mi.join(daily_nonmi, on="date", how="full", coalesce=True)
    .with_columns(pl.col("mi").fill_null(0), pl.col("non_mi").fill_null(0))
    .sort("date")
    .with_columns(
        pl.col("mi").rolling_mean(14).alias("mi_14d_avg"),
        pl.col("non_mi").rolling_mean(14).alias("non_mi_14d_avg"),
    )
)

daily_their_prs = (
    df_their_prs.with_columns(pl.col("closed_at").dt.truncate("1d").alias("date"))
    .group_by("date")
    .agg(pl.len().alias("prs_merged"))
    .sort("date")
    .with_columns(pl.col("prs_merged").rolling_mean(14).alias("prs_merged_14d_avg"))
)

daily_their_prs

In [ ]:
# https://mathlib-initiative.zulipchat.com/#narrow/channel/522994-general/topic/Mathlib.20Initiative.20statistics/with/585903063
# number of PRs commented on by reviewers since MI reviewer team started
df_team_prs = df_prs.filter(expr_commenters_include_any(logins))
pdf_team = df_team_prs.to_pandas()
pdf_team = pdf_team[
    (pdf_team["gh_created_at"] >= datetime(2025, 11, 1, tzinfo=timezone.utc))
    & (pdf_team["gh_created_at"] < datetime(2026, 4, 15, tzinfo=timezone.utc))
]
len(pdf_team)

In [ ]:
# Plot: raw daily count + 14-day average, since 2023-01-01
pdf = daily_their_prs.to_pandas()
pdf = pdf[pdf["date"] >= datetime(2023, 1, 1, tzinfo=timezone.utc)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(
    pdf["date"],
    pdf["prs_merged"],
    alpha=0.3,
    label="daily count",
    color="steelblue",
    width=1,
)
ax.plot(
    pdf["date"],
    pdf["prs_merged_14d_avg"],
    label="14-day avg",
    color="steelblue",
    linewidth=2,
)
ax.set_xlabel("date (UTC)")
ax.set_ylabel("PRs merged")
ax.set_title("MI-related PRs merged per day, since 2023-01-01)")
ax.legend()
fig.autofmt_xdate(rotation=45)
fig.tight_layout()
plt.show()

In [ ]:
# Plot: feat vs non-feat 14-day avg, since 2023-01-01
pdf = daily_mi_nonmi.to_pandas()
pdf = pdf[pdf["date"] >= datetime(2023, 1, 1, tzinfo=timezone.utc)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    pdf["date"],
    pdf["mi_14d_avg"],
    label="MI (14d avg)",
    color="#2e0050",
    linewidth=2,
)
ax.plot(
    pdf["date"],
    pdf["non_mi_14d_avg"],
    label="non-MI (14d avg)",
    color="#FF0073",
    linewidth=2,
)
ax.set_xlabel("date (UTC)")
ax.set_ylabel("PRs merged per day")
ax.set_title("PRs merged per day: MI vs non-MI (14d avg, since 2023-01-01)")
ax.legend()
fig.autofmt_xdate(rotation=45)
fig.tight_layout()
plt.show()